# A002_togashi_rogii_fast_local_gr_alignment

## ROGII — Fast Local GR Alignment

A001 failed because all wells were handled by `trend_fallback_too_long`, so the actual GR alignment never ran.

This A002 version removes that failure mode.

### Core idea

For each hidden evaluation row:

1. Start from a visible `TVT_input` trend anchor.
2. Search a local TVT window around that trend.
3. Compare horizontal-well GR to typewell GR at each candidate TVT.
4. Add a prior penalty so predictions do not drift too far from the visible trend.
5. Smooth the selected offset path per well.
6. Write `submission.csv`.

This is still a distinct approach from anchor-candidate selection and micro-blending, but it is much faster than beam search.

In [1]:
import os
import glob
import json
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RUN_ID = "A002"
NOTEBOOK_ID = "A002_togashi_rogii_fast_local_gr_alignment"
STRATEGY = "fast local GR alignment: visible TVT_input trend + per-row typewell GR matching + smoothed offset path"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT_PATH = WORK / "submission.csv"
DIAG_PATH = WORK / "diagnostics_A002_fast_local_gr_alignment.csv"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Notebook:", NOTEBOOK_ID)
print("Output:", OUT_PATH)

Notebook: A002_togashi_rogii_fast_local_gr_alignment
Output: /kaggle/working/submission.csv


In [2]:
class CFG:
    SEARCH_RADIUS = float(os.environ.get("ROGII_A002_SEARCH_RADIUS", "160.0"))
    GRID_STEP = float(os.environ.get("ROGII_A002_GRID_STEP", "1.0"))

    W_GR = float(os.environ.get("ROGII_A002_W_GR", "1.00"))
    W_DGR = float(os.environ.get("ROGII_A002_W_DGR", "0.08"))
    W_PRIOR = float(os.environ.get("ROGII_A002_W_PRIOR", "0.010"))

    OFFSET_SMOOTH_WIN = int(os.environ.get("ROGII_A002_OFFSET_SMOOTH_WIN", "151"))
    OFFSET_BLEND = float(os.environ.get("ROGII_A002_OFFSET_BLEND", "0.85"))

    SLOPE_TAIL_N = int(os.environ.get("ROGII_A002_SLOPE_TAIL_N", "300"))
    SLOPE_MIN_POINTS = int(os.environ.get("ROGII_A002_SLOPE_MIN_POINTS", "20"))
    SLOPE_CLIP = (-1.5, 1.5)

    MAX_OFFSET_ABS = float(os.environ.get("ROGII_A002_MAX_OFFSET_ABS", "140.0"))
    EPS = 1e-9

print({
    "SEARCH_RADIUS": CFG.SEARCH_RADIUS,
    "GRID_STEP": CFG.GRID_STEP,
    "OFFSET_SMOOTH_WIN": CFG.OFFSET_SMOOTH_WIN,
    "OFFSET_BLEND": CFG.OFFSET_BLEND,
    "MAX_OFFSET_ABS": CFG.MAX_OFFSET_ABS,
})

{'SEARCH_RADIUS': 160.0, 'GRID_STEP': 1.0, 'OFFSET_SMOOTH_WIN': 151, 'OFFSET_BLEND': 0.85, 'MAX_OFFSET_ABS': 140.0}


In [3]:
def find_data_root():
    candidates = [
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path("."),
    ]
    for root in candidates:
        if (root / "sample_submission.csv").exists() and (root / "test").exists():
            return root

    for f in glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True):
        root = Path(f).parent
        if (root / "test").exists():
            return root

    raise FileNotFoundError("Could not find ROGII competition data. Attach the competition dataset.")

DATA = find_data_root()
TEST_DIR = DATA / "test"
SAMPLE_PATH = DATA / "sample_submission.csv"

print("DATA:", DATA)
print("TEST:", TEST_DIR)
print("SAMPLE:", SAMPLE_PATH)

DATA: /kaggle/input/competitions/rogii-wellbore-geology-prediction
TEST: /kaggle/input/competitions/rogii-wellbore-geology-prediction/test
SAMPLE: /kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv


In [4]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def robust_center_scale(x):
    x = np.asarray(x, dtype=float)
    ok = np.isfinite(x)
    if ok.sum() == 0:
        return 0.0, 1.0
    med = float(np.nanmedian(x[ok]))
    mad = float(np.nanmedian(np.abs(x[ok] - med)))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale < CFG.EPS:
        scale = float(np.nanstd(x[ok]))
    if not np.isfinite(scale) or scale < CFG.EPS:
        scale = 1.0
    return med, scale


def robust_z(x, center, scale):
    return (np.asarray(x, dtype=float) - center) / max(scale, CFG.EPS)


def interp_safe(x, xp, fp):
    x = np.asarray(x, dtype=float)
    xp = np.asarray(xp, dtype=float)
    fp = np.asarray(fp, dtype=float)

    keep = np.isfinite(xp) & np.isfinite(fp)
    xp = xp[keep]
    fp = fp[keep]

    if len(xp) == 0:
        return np.zeros_like(x, dtype=float)
    if len(xp) == 1:
        return np.full_like(x, fp[0], dtype=float)

    order = np.argsort(xp)
    xp = xp[order]
    fp = fp[order]
    return np.interp(x, xp, fp, left=fp[0], right=fp[-1])


def parse_sample(sample):
    out = sample.copy()
    out["id"] = out["id"].astype(str)
    out["well"] = out["id"].str[:8]
    out["row_idx"] = out["id"].str.split("_").str[-1].astype(int)
    return out


def load_test_well(well):
    hw_path = TEST_DIR / f"{well}__horizontal_well.csv"
    tw_path = TEST_DIR / f"{well}__typewell.csv"
    if not hw_path.exists():
        raise FileNotFoundError(hw_path)
    if not tw_path.exists():
        raise FileNotFoundError(tw_path)
    return pd.read_csv(hw_path), pd.read_csv(tw_path)


def make_odd(n):
    n = int(n)
    if n < 3:
        n = 3
    return n if n % 2 == 1 else n + 1

In [5]:
def estimate_slope(hw, eval_row_idxs):
    if "TVT_input" not in hw.columns or "MD" not in hw.columns:
        return 0.0

    tvt = pd.to_numeric(hw["TVT_input"], errors="coerce").to_numpy(dtype=float)
    md = pd.to_numeric(hw["MD"], errors="coerce").to_numpy(dtype=float)

    eval_min = int(np.min(eval_row_idxs)) if len(eval_row_idxs) else len(hw)
    idx = np.arange(len(hw))
    known = np.isfinite(tvt) & np.isfinite(md) & (idx < eval_min)
    known_idx = np.where(known)[0]

    if len(known_idx) < CFG.SLOPE_MIN_POINTS:
        return 0.0

    tail_idx = known_idx[-CFG.SLOPE_TAIL_N:]
    x = md[tail_idx]
    y = tvt[tail_idx]
    ok = np.isfinite(x) & np.isfinite(y)

    if ok.sum() < CFG.SLOPE_MIN_POINTS or np.nanstd(x[ok]) < CFG.EPS:
        return 0.0

    slope = float(np.polyfit(x[ok], y[ok], 1)[0])
    return float(np.clip(slope, CFG.SLOPE_CLIP[0], CFG.SLOPE_CLIP[1]))


def last_anchor(hw, eval_row_idxs, tw_tvt=None):
    tvt = pd.to_numeric(hw.get("TVT_input", pd.Series([np.nan] * len(hw))), errors="coerce").to_numpy(dtype=float)
    md = pd.to_numeric(hw.get("MD", pd.Series(np.arange(len(hw)))), errors="coerce").to_numpy(dtype=float)

    eval_min = int(np.min(eval_row_idxs)) if len(eval_row_idxs) else len(hw)
    idx = np.arange(len(hw))
    known = np.isfinite(tvt) & np.isfinite(md) & (idx < eval_min)

    if known.sum() > 0:
        j = np.where(known)[0][-1]
        return float(md[j]), float(tvt[j]), int(j)

    fallback_tvt = float(np.nanmedian(tw_tvt)) if tw_tvt is not None and len(tw_tvt) else 0.0
    j = max(0, eval_min - 1)
    return float(md[j]) if len(md) else 0.0, fallback_tvt, int(j)


def trend_prediction(hw, tw, eval_row_idxs, reason):
    if "MD" not in hw.columns:
        hw["MD"] = np.arange(len(hw), dtype=float)

    md = pd.to_numeric(hw["MD"], errors="coerce").to_numpy(dtype=float)
    tw_tvt = None
    if "TVT" in tw.columns:
        tw_tvt = pd.to_numeric(tw["TVT"], errors="coerce").dropna().to_numpy(dtype=float)

    slope = estimate_slope(hw, eval_row_idxs)
    last_md, last_tvt, last_idx = last_anchor(hw, eval_row_idxs, tw_tvt)
    pred = last_tvt + slope * (md[eval_row_idxs] - last_md)

    if tw_tvt is not None and len(tw_tvt):
        pred = np.clip(pred, np.nanmin(tw_tvt), np.nanmax(tw_tvt))

    diag = {
        "status": f"trend_fallback_{reason}",
        "slope": float(slope),
        "last_tvt": float(last_tvt),
        "last_idx": int(last_idx),
        "n_eval": int(len(eval_row_idxs)),
        "offset_mean": 0.0,
        "offset_p95_abs": 0.0,
        "local_uncertainty_mean": np.nan,
        "tvt_min": float(np.min(pred)) if len(pred) else np.nan,
        "tvt_mean": float(np.mean(pred)) if len(pred) else np.nan,
        "tvt_max": float(np.max(pred)) if len(pred) else np.nan,
    }
    return pred, diag

In [6]:
def local_gr_alignment(hw, tw, eval_row_idxs):
    eval_row_idxs = np.asarray(eval_row_idxs, dtype=int)
    n = len(eval_row_idxs)

    if n == 0:
        return np.array([], dtype=float), {"status": "empty"}

    if "MD" not in hw.columns:
        hw["MD"] = np.arange(len(hw), dtype=float)
    if "GR" not in hw.columns or "TVT" not in tw.columns or "GR" not in tw.columns:
        return trend_prediction(hw, tw, eval_row_idxs, "missing_columns")

    hw_md = pd.to_numeric(hw["MD"], errors="coerce").to_numpy(dtype=float)
    hw_gr_raw = pd.to_numeric(hw["GR"], errors="coerce").to_numpy(dtype=float)

    tw2 = tw[["TVT", "GR"]].copy()
    tw2["TVT"] = pd.to_numeric(tw2["TVT"], errors="coerce")
    tw2["GR"] = pd.to_numeric(tw2["GR"], errors="coerce")
    tw2 = tw2.dropna().sort_values("TVT")

    if len(tw2) < 5:
        return trend_prediction(hw, tw, eval_row_idxs, "short_typewell")

    tw_tvt = tw2["TVT"].to_numpy(dtype=float)
    tw_gr_raw = tw2["GR"].to_numpy(dtype=float)

    center, scale = robust_center_scale(np.r_[tw_gr_raw, hw_gr_raw[np.isfinite(hw_gr_raw)]])
    tw_gr = robust_z(tw_gr_raw, center, scale)

    hw_gr = robust_z(hw_gr_raw, center, scale)
    hw_gr = (
        pd.Series(hw_gr)
        .interpolate(limit_direction="both")
        .rolling(9, min_periods=1, center=True)
        .median()
        .to_numpy(dtype=float)
    )

    tw_dgr = np.gradient(tw_gr, tw_tvt)
    tw_dgr = np.nan_to_num(tw_dgr, nan=0.0, posinf=0.0, neginf=0.0)

    hw_dgr = np.gradient(hw_gr, hw_md)
    hw_dgr = np.nan_to_num(hw_dgr, nan=0.0, posinf=0.0, neginf=0.0)

    slope = estimate_slope(hw, eval_row_idxs)
    last_md, last_tvt, last_idx = last_anchor(hw, eval_row_idxs, tw_tvt)

    eval_md = hw_md[eval_row_idxs]
    init_tvt = last_tvt + slope * (eval_md - last_md)
    init_tvt = np.clip(init_tvt, np.nanmin(tw_tvt), np.nanmax(tw_tvt))

    offsets = np.arange(-CFG.SEARCH_RADIUS, CFG.SEARCH_RADIUS + CFG.GRID_STEP, CFG.GRID_STEP, dtype=float)
    n_grid = len(offsets)

    raw_offsets = np.zeros(n, dtype=float)
    local_unc = np.zeros(n, dtype=float)

    for i, row in enumerate(eval_row_idxs):
        cand_tvt = np.clip(init_tvt[i] + offsets, np.nanmin(tw_tvt), np.nanmax(tw_tvt))

        gr_here = interp_safe(cand_tvt, tw_tvt, tw_gr)
        dgr_here = interp_safe(cand_tvt, tw_tvt, tw_dgr)

        hgr = hw_gr[row]
        hdgr = hw_dgr[row]

        cost = (
            CFG.W_GR * (hgr - gr_here) ** 2
            + CFG.W_DGR * (hdgr - dgr_here * max(abs(slope), 0.05)) ** 2
            + CFG.W_PRIOR * (offsets / max(CFG.SEARCH_RADIUS, 1.0)) ** 2
        )

        order = np.argsort(cost)
        best = int(order[0])
        second = int(order[1]) if len(order) > 1 else best

        raw_offsets[i] = offsets[best]
        local_unc[i] = float(cost[second] - cost[best]) if second != best else np.nan

    win = make_odd(min(CFG.OFFSET_SMOOTH_WIN, max(3, n // 3 * 2 + 1)))
    smoothed_offsets = (
        pd.Series(raw_offsets)
        .rolling(win, min_periods=1, center=True)
        .median()
        .rolling(win, min_periods=1, center=True)
        .mean()
        .to_numpy(dtype=float)
    )

    final_offsets = CFG.OFFSET_BLEND * smoothed_offsets + (1.0 - CFG.OFFSET_BLEND) * raw_offsets
    final_offsets = np.clip(final_offsets, -CFG.MAX_OFFSET_ABS, CFG.MAX_OFFSET_ABS)

    pred = init_tvt + final_offsets
    pred = np.clip(pred, np.nanmin(tw_tvt), np.nanmax(tw_tvt))

    diag = {
        "status": "local_gr_aligned",
        "slope": float(slope),
        "last_tvt": float(last_tvt),
        "last_idx": int(last_idx),
        "n_eval": int(n),
        "n_grid": int(n_grid),
        "offset_mean": float(np.mean(final_offsets)),
        "offset_p95_abs": float(np.percentile(np.abs(final_offsets), 95)),
        "offset_max_abs": float(np.max(np.abs(final_offsets))),
        "raw_offset_p95_abs": float(np.percentile(np.abs(raw_offsets), 95)),
        "local_uncertainty_mean": float(np.nanmean(local_unc)),
        "tvt_min": float(np.min(pred)),
        "tvt_mean": float(np.mean(pred)),
        "tvt_max": float(np.max(pred)),
    }
    return pred, diag

In [7]:
t0 = time.time()

sample = pd.read_csv(SAMPLE_PATH)
sample = parse_sample(sample)

parts = []
diagnostics = []
n_total_wells = sample["well"].nunique()

for i, (well, g) in enumerate(sample.groupby("well", sort=True), 1):
    row_idxs = g["row_idx"].to_numpy(dtype=int)

    try:
        hw, tw = load_test_well(well)
        pred, diag = local_gr_alignment(hw, tw, row_idxs)

        part = pd.DataFrame({
            "id": g["id"].values,
            "tvt": pred,
        })

        diag = dict(diag)
        diag["well"] = well
        diag["rows"] = int(len(part))
        diagnostics.append(diag)

    except Exception as e:
        part = g[["id", "tvt"]].copy()
        diagnostics.append({
            "well": well,
            "status": "sample_fallback_exception",
            "error": repr(e),
            "rows": int(len(part)),
            "slope": np.nan,
            "last_tvt": np.nan,
            "n_eval": int(len(part)),
            "offset_mean": np.nan,
            "offset_p95_abs": np.nan,
            "local_uncertainty_mean": np.nan,
            "tvt_min": float(part["tvt"].min()) if len(part) else np.nan,
            "tvt_mean": float(part["tvt"].mean()) if len(part) else np.nan,
            "tvt_max": float(part["tvt"].max()) if len(part) else np.nan,
        })

    parts.append(part)

    if i % 20 == 0:
        print(f"processed wells: {i}/{n_total_wells} | elapsed={time.time() - t0:.1f}s", flush=True)

sub = pd.concat(parts, axis=0, ignore_index=True)
sub = sample[["id"]].merge(sub, on="id", how="left")

if sub["tvt"].isna().any():
    fill_value = float(sample["tvt"].median()) if "tvt" in sample.columns and sample["tvt"].notna().any() else 0.0
    sub["tvt"] = sub["tvt"].fillna(fill_value)

diag_df = pd.DataFrame(diagnostics)

sub.to_csv(OUT_PATH, index=False)
diag_df.to_csv(DIAG_PATH, index=False)

elapsed = time.time() - t0

print("Saved:", OUT_PATH)
print("Saved:", DIAG_PATH)
print("elapsed_sec:", elapsed)
print("rows:", len(sub))
print("n_wells:", sub["id"].str[:8].nunique())
print("status counts:")
print(diag_df["status"].value_counts(dropna=False))
print("tvt summary:")
print(sub["tvt"].describe())
display(diag_df)

Saved: /kaggle/working/submission.csv
Saved: /kaggle/working/diagnostics_A002_fast_local_gr_alignment.csv
elapsed_sec: 3.38106107711792
rows: 14151
n_wells: 3
status counts:
status
local_gr_aligned    3
Name: count, dtype: int64
tvt summary:
count    14151.000000
mean     11909.788016
std        258.070322
min      11606.360089
25%      11637.483868
50%      11758.650048
75%      12201.715655
max      12240.627489
Name: tvt, dtype: float64


,status,slope,last_tvt,last_idx,n_eval,n_grid,offset_mean,offset_p95_abs,offset_max_abs,raw_offset_p95_abs,local_uncertainty_mean,tvt_min,tvt_mean,tvt_max,well,rows
0,local_gr_aligned,0.003365,11747.37,1441,3836,321,2.719839,26.243626,65.493377,65.0,0.011726,11731.924745,11756.545420,11822.476927,000d7d20,3836
1,local_gr_aligned,-0.008528,12223.54,1544,6014,321,6.759050,32.538659,68.557285,71.0,0.001054,12137.192655,12204.650150,12240.627489,00bbac68,6014
2,local_gr_aligned,0.024860,11604.82,2082,4301,321,-24.130906,69.958940,74.914735,72.0,0.001479,11606.360089,11634.163214,11676.657632,00e12e8b,4301


In [8]:
summary = {
    "notebook": NOTEBOOK_ID,
    "run_id": RUN_ID,
    "strategy": STRATEGY,
    "data_root": str(DATA),
    "output_path": str(OUT_PATH),
    "diagnostics_path": str(DIAG_PATH),
    "submission_sha": sha256_file(OUT_PATH),
    "rows": int(len(sub)),
    "n_wells": int(sub["id"].str[:8].nunique()),
    "elapsed_sec": float(elapsed),
    "controls": {
        "search_radius": CFG.SEARCH_RADIUS,
        "grid_step": CFG.GRID_STEP,
        "w_gr": CFG.W_GR,
        "w_dgr": CFG.W_DGR,
        "w_prior": CFG.W_PRIOR,
        "offset_smooth_win": CFG.OFFSET_SMOOTH_WIN,
        "offset_blend": CFG.OFFSET_BLEND,
        "max_offset_abs": CFG.MAX_OFFSET_ABS,
        "slope_tail_n": CFG.SLOPE_TAIL_N,
        "slope_clip": CFG.SLOPE_CLIP,
    },
    "tvt_min": float(sub["tvt"].min()),
    "tvt_mean": float(sub["tvt"].mean()),
    "tvt_max": float(sub["tvt"].max()),
    "status_counts": diag_df["status"].value_counts(dropna=False).to_dict() if "status" in diag_df.columns else {},
}

print("===== COPY THIS TO GPT START =====")
print(f"# {RUN_ID} Run Summary")
print(f"- strategy: `{summary['strategy']}`")
print(f"- submission_sha: `{summary['submission_sha']}`")
print(f"- rows: `{summary['rows']}`")
print(f"- n_wells: `{summary['n_wells']}`")
print(f"- elapsed_sec: `{summary['elapsed_sec']:.1f}`")
print(f"- tvt_min: `{summary['tvt_min']:.6f}`")
print(f"- tvt_mean: `{summary['tvt_mean']:.6f}`")
print(f"- tvt_max: `{summary['tvt_max']:.6f}`")
print()
print(f"## {RUN_ID} status counts")
print(diag_df["status"].value_counts(dropna=False).to_string())
print()
print(f"## {RUN_ID} diagnostics")
print(diag_df.to_string(index=False))
print()
print(f"## {RUN_ID} manifest")
print(json.dumps(summary, indent=2))
print("===== COPY THIS TO GPT END =====")

===== COPY THIS TO GPT START =====
# A002 Run Summary
- strategy: `fast local GR alignment: visible TVT_input trend + per-row typewell GR matching + smoothed offset path`
- submission_sha: `ef9e3164a8f92fa0042264ef5fd49e03e84fd3df26985783e05084043a96f0a7`
- rows: `14151`
- n_wells: `3`
- elapsed_sec: `3.4`
- tvt_min: `11606.360089`
- tvt_mean: `11909.788016`
- tvt_max: `12240.627489`

## A002 status counts
status
local_gr_aligned    3

## A002 diagnostics
          status     slope  last_tvt  last_idx  n_eval  n_grid  offset_mean  offset_p95_abs  offset_max_abs  raw_offset_p95_abs  local_uncertainty_mean      tvt_min     tvt_mean      tvt_max     well  rows
local_gr_aligned  0.003365  11747.37      1441    3836     321     2.719839       26.243626       65.493377                65.0                0.011726 11731.924745 11756.545420 11822.476927 000d7d20  3836
local_gr_aligned -0.008528  12223.54      1544    6014     321     6.759050       32.538659       68.557285                71.0 

## A002 interpretation

A002 should not show `trend_fallback_too_long`.

If it still scores poorly, the lesson is useful:

- pure GR-to-typewell alignment is not enough;
- visible-prefix trend alone is weak;
- this approach needs either candidate anchoring, spatial priors, or a validation-gated overlay.

That would make A002 a valid negative Working Note experiment.